# 1 Tworzenie tematu kafka

In [ ]:
# wynik komendy --list
# transactions

# 2 Tworzenie producenta

In [1]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(0.5)

producer.flush()
producer.close()

Overwriting producer.py


# 3 Tworzenie konsumentów bezstanowych - filtrowanie i wzbogacanie

In [3]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Nasłuchuję na duże transakcje (amount > 3000)...")

for message in consumer:
    dane = message.value
    amount = dane.get('amount', 0)
    if amount > 3000:
        tx_id = dane.get('tx_id', 'Brak ID')
        city = dane.get('store', 'Nieznane miasto')
        category = dane.get('category', 'Brak kategorii')
        
        print(f"ALERT: {tx_id} | {amount} PLN | {city} | {category}")

Overwriting consumer_filter.py


In [4]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='risk_group', 
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)
print("Rozpoczęto wzbogacanie transakcji o poziom ryzyka...")

for message in consumer:
    dane = message.value
    amount = dane.get('amount', 0)
    
    if amount > 3000:
        dane['risk_level'] = "HIGH"
    elif amount > 1000:
        dane['risk_level'] = "MEDIUM"
    else:
        dane['risk_level'] = "LOW"
        
    print(dane)
    

Overwriting consumer_enrich.py


# 4 Tworzenie konsumentów stanowych - agregacja per sklep i per katagoria

In [5]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = {}
total_msgs_processed = 0
current_batch_count = 0

print("Rozpoczęto zliczanie statystyk per sklep (paczki po 10 wiadomości)...")

for message in consumer:
    transaction = message.value
    
    store = transaction.get('store', 'Nieznany')
    amount = transaction.get('amount', 0)
    store_counts[store] += 1
    total_amount[store] = total_amount.get(store, 0) + amount
    
    current_batch_count += 1
    total_msgs_processed += 1
    
    if current_batch_count == 10:
        print(f"\n--- Podsumowanie dla ostatnich 10 wiadomości (łącznie: {total_msgs_processed}) ---")
        print(f"{'Sklep':<15} | {'Liczba':<8} | {'Suma':<10} | {'Średnia':<10}")
        print("-" * 52)
        
        for s, count in store_counts.items():
            suma = total_amount[s]
            srednia = suma / count if count > 0 else 0
            print(f"{s:<15} | {count:<8} | {suma:<10.2f} | {srednia:<10.2f}")

        store_counts.clear()
        total_amount.clear()
        current_batch_count = 0

Overwriting consumer_count.py


In [6]:
%%file consumer_stats.py
from kafka import KafkaConsumer
from collections import defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='category_stats_group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

# Słownik automatycznie tworzący strukturę statystyk dla nowej kategorii
category_stats = defaultdict(lambda: {'count': 0, 'sum': 0.0, 'min': float('inf'), 'max': float('-inf')})

total_msgs_processed = 0
current_batch_count = 0

print("Rozpoczęto śledzenie statystyk per kategoria (tumbling window: 10 wiadomości)...")

for message in consumer:
    transaction = message.value
    
    category = transaction.get('category', 'Nieznana')
    amount = transaction.get('amount', 0.0)
    
    # 1. Pobranie i aktualizacja statystyk dla danej kategorii w bieżącym oknie
    stats = category_stats[category]
    stats['count'] += 1
    stats['sum'] += amount
    
    if amount < stats['min']:
        stats['min'] = amount
    if amount > stats['max']:
        stats['max'] = amount
        
    current_batch_count += 1
    total_msgs_processed += 1
    
    # 2. Sprawdzenie, czy okno się wypełniło
    if current_batch_count == 10:
        print(f"\n--- Statystyki dla ostatnich 10 wiadomości (łącznie przetworzono: {total_msgs_processed}) ---")
        print(f"{'Kategoria':<15} | {'Liczba':<8} | {'Przychód':<12} | {'Min':<8} | {'Max':<8}")
        print("-" * 61)
        
        for cat, data in category_stats.items():
            # Zabezpieczenie przed dziwnym formatowaniem, gdyby min/max zostało jako nieskończoność
            # (choć w tym przypadku dla wyczyszczonego słownika count będzie > 0)
            print(f"{cat:<15} | {data['count']:<8} | {data['sum']:<12.2f} | {data['min']:<8.2f} | {data['max']:<8.2f}")
            
        # 3. KLUCZOWY KROK: Resetowanie okna (tumbling window)
        category_stats.clear()
        current_batch_count = 0

Overwriting consumer_stats.py


# 5 Uruchamianie wielu konsumentow

In [ ]:
# Odpowiedzi na pytania: 
# 1. Konsument będzie w trybie nasłuchiwania ale jako iż nie będzie żadnych nowych transakcji generowanych przez producenta to nic się nie wydarzy
# 2. W tym przypadku tylko jeden konsument będzie otrzymywał dane
# 3. W przetwarzaniu bezstanowym wynik zależy tylko od aktualnej wiadomosci, natomiast w przetwarzaniu stanowym wynik zależy nie tylko od aktualnej wiadomości ale i od historycznych.

# 6 Praca domowa

In [7]:
%%file consumer_velocity_anomaly.py
from kafka import KafkaConsumer
from collections import defaultdict
import json
import time

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='velocity_anomaly_group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

user_history = defaultdict(list)
TIME_WINDOW_SEC = 60
MAX_TRANSACTIONS = 3

print(f"Nasłuchiwanie anomalii: > {MAX_TRANSACTIONS} transakcje w ciągu {TIME_WINDOW_SEC}s...")

for message in consumer:
    transaction = message.value
    user_id = transaction.get('user_id')
    if not user_id:
        continue
    current_time = time.time()
    user_history[user_id].append(current_time)

    user_history[user_id] = [
        t for t in user_history[user_id] 
        if current_time - t <= TIME_WINDOW_SEC
    ]
    if len(user_history[user_id]) > MAX_TRANSACTIONS:
        ilosc = len(user_history[user_id])
        print(f"[ALERT] Użytkownik {user_id} wykonał {ilosc} transakcji w ciągu ostatnich 60 sekund!")


Overwriting consumer_velocity_anomaly.py
